# Multimodal Story Generation (Multi-Scene Edition)

**Author:** AI Assistant
**Description:** This notebook implements a Multimodal Story Generator that creates **consistent, multi-scene stories**.
1. **GPT-4o-mini** (OpenAI): Generates a story split into 8 scenes with consistent character descriptions.
2. **BLIP-2** (Hugging Face): Analyzes initial inspiration images.
3. **DreamShaper-8** (Hugging Face): Generates high-quality art for *each* scene.

**Instructions:**
1. Run the "Setup" cells.
2. Enter API Key.
3. Run the App.

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q opencv-python transformers accelerate bitsandbytes diffusers gradio openai safetensors

## 2. Imports & Configuration

In [ ]:
import os
import torch
import json
from PIL import Image
from diffusers import StableDiffusionPipeline, EulerDiscreteScheduler
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from openai import OpenAI
import gradio as gr
import base64
from io import BytesIO

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: Running on CPU will be extremely slow. Please enable GPU in Runtime > Change runtime type.")

In [ ]:
# Secure API Key Input
from getpass import getpass
OPENAI_API_KEY = getpass("Enter your OpenAI API Key: ").strip()

client = OpenAI(api_key=OPENAI_API_KEY)

## 3. Load AI Models (Images)

In [ ]:
print("Loading BLIP-2 (Image Captioning)...")
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16
)
blip_model.to(device)
print("BLIP-2 Loaded.")

print("Loading Stable Diffusion (DreamShaper-8)...")
# Switched to DreamShaper 8 (Much better artistic quality than base SD 1.5)
model_id = "Lykon/dreamshaper-8"
scheduler = EulerDiscreteScheduler.from_pretrained(model_id, subfolder="scheduler")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id, scheduler=scheduler, torch_dtype=torch.float16
)
pipe.to(device)
print("Stable Diffusion Loaded.")

## 4. Helper Functions

In [ ]:
def image_to_base64(img):
    """Converts PIL Image to Base64 string for HTML display."""
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{img_str}"

def analyze_image(image):
    """Generates a description for an uploaded image using BLIP-2."""
    if image is None:
        return None
    
    inputs = processor(images=image, return_tensors="pt").to(device, torch.float16)
    generated_ids = blip_model.generate(**inputs)
    description = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return description

def generate_story_scenes(topic, image_context=None):
    """Generates a JSON containing story scenes using GPT-4o-mini."""
    
    system_prompt = """You are a creative storyteller. 
    You must split your story into 8 distinct scenes. 
    You must write DETAILED, DESCRIPTIVE paragraphs (at least 80-100 words) for each scene.
    
    STRICT IMAGE RULES:
    1. You must alternate between 'Character Shots' and 'Landscape/Wide Shots'.
    2. At least 4 scenes MUST be pure landscapes with NO characters visible.
    3. For landscape scenes, purely describe the environment (e.g., "A vast golden desert under a purple sky"). DO NOT MENTION THE CHARACTER in the image_prompt.
    4. For character scenes, use the placeholder '[CHARACTER]' (e.g. "[CHARACTER] standing on a rock").
    
    Output strictly valid JSON."""
    
    user_prompt = f"Write a short story about: '{topic}'. Make sure to visualize the world itself in many scenes without the character."
    if image_context:
        user_prompt += f"Incorporate this visual context: '{image_context}'. "
        
    user_prompt += """
    Return ONLY a JSON object with this structure:
    {
        "title": "Story Title",
        "consistent_visual_style": "A description of the art style, e.g. 'masterpiece, best quality, 8k, cinematic lighting, pixar style, detailed'.",
        "character_design": "A detailed visual description of the main character (e.g. 'young woman with red hair wearing a space suit').",
        "scenes": [
            {"text": "Story paragraph 1...", "image_prompt": "A majestic waterfall cascading down purple rocks"},
            {"text": "Story paragraph 2...", "image_prompt": "[CHARACTER] drinking from the river"}
        ]
    }
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={ "type": "json_object" }
    )
    
    try:
        content = json.loads(response.choices[0].message.content)
        return content
    except Exception as e:
        print(f"JSON Error: {e}")
        return None

def generate_image_sd(style_prompt, character_design, scene_action):
    """Combines consistent style + character design (if [CHARACTER] is used) + action."""
    
    if "[CHARACTER]" in scene_action:
        # Replace placeholder with specific character description
        final_action = scene_action.replace("[CHARACTER]", character_design)
    else:
        # Scene without character
        final_action = scene_action
        
    # Combine: Action (with or without character description) + Style
    final_prompt = f"{final_action}, {style_prompt}"
    final_prompt = final_prompt[:350]
    
    # Negative prompts remove bad artifacts
    negative_prompt = "bad quality, worst quality, low resolution, blurry, distorted, deformed hands, ugly, watermark, text, varying character features"
    
    image = pipe(
        prompt=final_prompt, 
        negative_prompt=negative_prompt, 
        num_inference_steps=30,  # Slightly more steps for quality
        guidance_scale=7.5
    ).images[0]
    return image

## 5. Main Loop & UI

**IMPORTANT:** After running the cell below, wait for the output to show a link starting with `https://` and ending with `.gradio.live`. **Click that link** to open the story generator in a new, full-screen tab. This is perfect for sharing or presenting!

In [ ]:
def process_pipeline(user_topic, user_image):
    # 1. Image Analysis
    image_context = None
    if user_image is not None:
        image_context = analyze_image(user_image)

    # 2. Generate Story Structure (JSON)
    story_json = generate_story_scenes(user_topic, image_context)
    
    if not story_json:
        return "<h1>Error generating story. Please try again.</h1>", None

    # 3. Build HTML Output
    html_output = f"<h1>{story_json.get('title', 'Generated Story')}</h1>"
    style_prompt = story_json.get("consistent_visual_style", "cinematic, 4k")
    character_design = story_json.get("character_design", "")
    
    scenes = story_json.get("scenes", [])
    
    for i, scene in enumerate(scenes):
        text = scene["text"]
        action_prompt = scene["image_prompt"]
        
        # Generate Image for this scene
        print(f"Generating image for Scene {i+1}...")
        # Pass the separate character design prompt
        img = generate_image_sd(style_prompt, character_design, action_prompt)
        img_b64 = image_to_base64(img)
        
        # Append to HTML (Text -> Image)
        html_output += f"""
        <div style='background-color: #222; padding: 20px; margin-bottom: 20px; border-radius: 10px;'>
            <p style='font-size: 18px; color: #ddd; margin-bottom: 15px;'>{text}</p>
            <img src='{img_b64}' style='width: 100%; max-width: 600px; border-radius: 8px;' />
        </div>
        """
    
    # 4. Save to File
    filename = "my_generated_story.html"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(html_output)
    
    return html_output, filename

# Gradio Interface
with gr.Blocks(title="Multimodal Story Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 Multimodal Story Generator")
    gr.Markdown("Generates a multi-scene story with consistent characters. (Text -> Image Loop)")
    
    with gr.Row():
        with gr.Column(scale=1):
            topic_input = gr.Textbox(label="Story Topic/Idea", placeholder="E.g. A cyber-cat detective in neon tokyo")
            image_input = gr.Image(type="pil", label="Initial Inspiration (Optional)")
            submit_btn = gr.Button("✨ Generate Story", variant="primary")
        
        with gr.Column(scale=2):
            download_btn = gr.File(label="📥 Download Story (HTML)")
            story_display = gr.HTML(label="Generated Story")

    submit_btn.click(
        fn=process_pipeline,
        inputs=[topic_input, image_input],
        outputs=[story_display, download_btn]
    )

demo.launch(share=True, debug=True)